# *Leucadendron argenteum* — Hyperspectral Spectral and Classification Analysis

## AVIRIS-NG Level-3 | BioSCape 2023 | Table Mountain National Park

This notebook contains the spectral analysis and classification stages of the study. It deliberately stops before any spatial prediction, LiDAR integration, or species mapping is performed.

The workflow begins with the original species-spectrum extraction file and **reconstructs the 324-band spectral library using the exact band-selection rule used in the dedicated band-selection notebook**. That single filtered library is then passed through the spectral profiles, Spectral Angle Mapper, PCA, classification models, bootstrap analysis, spectral-region ablation, and model interpretation.

### What this notebook does

1. Load the original extracted species spectra.
2. Apply the documented rule used to obtain the **324 retained wavelength bands**.
3. Save the resulting 324-band spectral library and retained-band table.
4. Vector-normalise the retained spectra.
5. Produce the manuscript-style spectral-profile figure for all species and the highlighted *L. argenteum* profile.
6. Calculate spectral-angle similarity and target-versus-species permutation tests.
7. Run PCA and visualise the reduced spectral space.
8. Compare logistic regression, linear SVM, RBF SVM, random forest, k-NN and XGBoost using 5-fold stratified cross-validation.
9. Calculate overall and *L. argenteum*-specific performance metrics from out-of-fold predictions.
10. Rank species by F1-score and examine the species most often confused with *L. argenteum*.
11. Bootstrap the out-of-fold predictions of the best-performing classifier to obtain 95% confidence intervals.
12. Test spectral-region ablation to examine how much classification performance depends on different wavelength regions.
13. Run wavelength-attribution analysis for XGBoost using SHAP where the package is available.
14. Write the tables and manuscript figures to the output directory.

> **Important:** 324 refers to the number of **retained spectral bands**, not the number of observations. All available spectra in the filtered library are passed to the downstream analysis, subject to the stated species/sample eligibility rule.

The band-selection stage is included here so there is one source of truth for the wavelength set used by the classification analysis.


## 1. Analysis settings

The settings below are deliberately kept near the beginning of the notebook. This makes the analysis easier to audit and makes it clear which decisions are analytical choices rather than hidden inside later code.

In [1]:
# Import the packages used throughout the notebook.

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

from scipy.stats import rankdata

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# XGBoost is part of the manuscript classifier comparison.
from xgboost import XGBClassifier

# SHAP is used later for wavelength-attribution analysis.
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False
    shap = None

print(f"SHAP available: {HAVE_SHAP}")

SHAP available: True


In [2]:
# Set the project paths and analysis settings.

ROOT = Path.cwd()

DATA_DIR = ROOT / "Data"

# This is the original spectral extraction file.
# The notebook rebuilds the 324-band library from this file.
RAW_SPECTRA_CSV = (
    DATA_DIR
    / "Hyperspectral"
    / "Outputs"
    / "species_spectra4.csv"
)

OUTPUT_DIR = (
    DATA_DIR
    / "Hyperspectral"
    / "Outputs_324bands"
)

FIG_DIR = OUTPUT_DIR / "Figures"
TABLE_DIR = OUTPUT_DIR / "Tables"

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# Species settings.

SPECIES_COLUMN = "Scientific.Names"
TARGET = "Leucadendron argenteum"


# The band-selection notebook gives 324 retained bands using the rule below.

EXPECTED_RETAINED_BANDS = 324

DROP_BELOW_NM = 450.0
DROP_ABOVE_NM = 2400.0

DROP_RANGES_NM = [
    (1340.0, 1480.0),
    (1800.0, 1980.0),
]

# The source band-selection notebook used NEG_RULE = "none".
NEGATIVE_REFLECTANCE_RULE = "none"

# Species with fewer than 20 retained spectra are excluded before
# constructing the analysis library, matching the source band-selection stage.
MIN_SAMPLES = 20


# Classification settings.

N_PCS = 10
CV_FOLDS = 5
RANDOM_STATE = 42
RF_TREES = 400
SAM_PERMUTATIONS = 999
BOOTSTRAP_REPS = 1000
ABLATION_REPEATS = 10


# Wavelength regions used later for leave-one-region-out analysis.

SPECTRAL_REGIONS = {
    "Visible": (400, 700),
    "Red-edge": (700, 780),
    "NIR": (780, 1350),
    "SWIR-1": (1450, 1800),
    "SWIR-2": (1970, 2500),
}

print(f"Raw spectral library : {RAW_SPECTRA_CSV}")
print(f"Target species       : {TARGET}")
print(f"Expected bands      : {EXPECTED_RETAINED_BANDS}")
print(f"Minimum samples      : {MIN_SAMPLES}")
print(f"Negative-value rule  : {NEGATIVE_REFLECTANCE_RULE}")

Raw spectral library : /Users/phemelo/Documents/LesHyperion/Data/Hyperspectral/Outputs/species_spectra4.csv
Target species       : Leucadendron argenteum
Expected bands      : 324
Minimum samples      : 20
Negative-value rule  : none


## 2. Build the 324-band spectral library

The analysis starts from the original species spectra extracted from the AVIRIS-NG Level-3 reflectance mosaics.

The retained wavelength set is reconstructed here rather than typed in manually. The rule is the same one used in the band-selection notebook:

- remove wavelengths below 450 nm;
- remove 1,340–1,480 nm;
- remove 1,800–1,980 nm;
- remove wavelengths above 2,400 nm;
- apply no additional negative-reflectance band filter.

The resulting wavelength set is checked against 324 bands. This means that every later analysis uses the same documented spectral library.


In [3]:
# Load the original spectral extraction file.

if not RAW_SPECTRA_CSV.exists():
    raise FileNotFoundError(
        f"Could not find the original spectral library at {RAW_SPECTRA_CSV}. "
        "Update RAW_SPECTRA_CSV in the settings cell."
    )

library_raw = pd.read_csv(RAW_SPECTRA_CSV)

if SPECIES_COLUMN not in library_raw.columns:
    raise KeyError(
        f"The spectral library must contain the '{SPECIES_COLUMN}' column."
    )

if TARGET not in library_raw[SPECIES_COLUMN].astype(str).values:
    raise ValueError(
        f"Target species '{TARGET}' was not found in the spectral library."
    )

print(
    f"Loaded {len(library_raw):,} observations "
    f"and {len(library_raw.columns):,} columns."
)

print(
    f"Target observations: "
    f"{(library_raw[SPECIES_COLUMN].astype(str) == TARGET).sum():,}"
)


Loaded 1,229 observations and 432 columns.
Target observations: 50


In [4]:
# Identify the delivered wavelength bands from their column names.

BAND_PATTERN = re.compile(r"^B\d+_(\d+)nm$")

band_info = []

for column in library_raw.columns:

    match = BAND_PATTERN.match(str(column))

    if match:
        band_info.append(
            (column, float(match.group(1)))
        )

if not band_info:
    raise ValueError(
        "No spectral columns matching B{number}_{wavelength}nm were found."
    )

# Sort by wavelength so that the spectral matrix follows the actual
# wavelength sequence rather than the original CSV column order.

band_info = sorted(
    band_info,
    key=lambda item: item[1]
)

band_cols_all = [
    column
    for column, _ in band_info
]

wl_all = np.array(
    [wavelength for _, wavelength in band_info],
    dtype=float
)

print(
    f"Delivered spectral bands: {len(band_cols_all)}"
)

print(
    f"Delivered wavelength range: "
    f"{wl_all.min():.0f}-{wl_all.max():.0f} nm"
)


Delivered spectral bands: 425
Delivered wavelength range: 377-2501 nm


### 2.1 Apply the band-selection rule

The next cells reproduce the band-selection stage that produced the 324-band library.

The species eligibility rule is applied first, as in the source notebook. The per-species sampling table and locality summary are deliberately omitted here because they are not part of the classification workflow.


In [5]:
# Keep only species with at least 20 extracted spectra.

species_counts = (
    library_raw[SPECIES_COLUMN]
    .astype(str)
    .value_counts()
)

eligible_species_for_library = species_counts[
    species_counts >= MIN_SAMPLES
].index

library_for_band_selection = library_raw[
    library_raw[SPECIES_COLUMN]
    .astype(str)
    .isin(eligible_species_for_library)
].reset_index(drop=True)

X_all = (
    library_for_band_selection[band_cols_all]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(float)
)

print(
    f"Observations entering band selection: "
    f"{len(library_for_band_selection):,}"
)

print(
    f"Species retained for the library: "
    f"{len(eligible_species_for_library)}"
)


Observations entering band selection: 1,229
Species retained for the library: 26


In [6]:
# Apply the exact wavelength rule that gives the 324 retained bands.

steps = []

keep = np.ones(
    len(band_cols_all),
    dtype=bool
)

steps.append(
    ("delivered", int(keep.sum()))
)


# Remove wavelengths below 450 nm.

keep &= ~(
    wl_all < DROP_BELOW_NM
)

steps.append(
    (
        f"after dropping < {DROP_BELOW_NM:.0f} nm",
        int(keep.sum())
    )
)


# Remove the two broad water-absorption windows.

for lower, upper in DROP_RANGES_NM:

    keep &= ~(
        (wl_all >= lower)
        &
        (wl_all <= upper)
    )

    steps.append(
        (
            f"after dropping {lower:.0f}-{upper:.0f} nm",
            int(keep.sum())
        )
    )


# Remove wavelengths above 2,400 nm.

keep &= ~(
    wl_all > DROP_ABOVE_NM
)

steps.append(
    (
        f"after dropping > {DROP_ABOVE_NM:.0f} nm",
        int(keep.sum())
    )
)


# The source band-selection notebook did not apply a negative-value band
# filter, so no such filter is added here.

if NEGATIVE_REFLECTANCE_RULE != "none":
    raise ValueError(
        "NEGATIVE_REFLECTANCE_RULE must be 'none' to reproduce "
        "the source 324-band library."
    )


# Store the retained wavelengths and their original column names.

good_indices = np.flatnonzero(keep)

band_cols_good = [
    band_cols_all[index]
    for index in good_indices
]

wl_good = wl_all[keep]


for label, number in steps:

    print(
        f"{label:45s} {number:3d} bands"
    )


print(
    f"\nRetained bands: {len(band_cols_good)}"
)

print(
    f"Retained wavelength range: "
    f"{wl_good.min():.0f}-{wl_good.max():.0f} nm"
)


# Stop rather than silently changing the wavelength set if the source
# library does not reproduce the expected 324 bands.

if len(band_cols_good) != EXPECTED_RETAINED_BANDS:

    raise ValueError(
        f"The band-selection rule produced "
        f"{len(band_cols_good)} bands, not "
        f"{EXPECTED_RETAINED_BANDS}. "
        "Check the source spectral library and the band-selection settings "
        "before continuing."
    )


# Build the 324-band library that all downstream analyses will use.

metadata_columns = [
    column
    for column in library_for_band_selection.columns
    if column not in band_cols_all
]

filtered_library = pd.concat(
    [
        library_for_band_selection[metadata_columns],
        library_for_band_selection[band_cols_good],
    ],
    axis=1
)

print(
    f"\n324-band library: "
    f"{len(filtered_library):,} observations × "
    f"{len(band_cols_good)} bands"
)


delivered                                     425 bands
after dropping < 450 nm                       410 bands
after dropping 1340-1480 nm                   382 bands
after dropping 1800-1980 nm                   345 bands
after dropping > 2400 nm                      324 bands

Retained bands: 324
Retained wavelength range: 452-2396 nm

324-band library: 1,229 observations × 324 bands


In [7]:
# Save the 324-band library and the retained wavelength list.

filtered_library_path = (
    TABLE_DIR
    / "spectral_library_filtered_324bands.csv"
)

retained_bands_path = (
    TABLE_DIR
    / "retained_bands_324.csv"
)

filtered_library.to_csv(
    filtered_library_path,
    index=False
)

pd.DataFrame({
    "band_column": band_cols_good,
    "wavelength_nm": wl_good,
}).to_csv(
    retained_bands_path,
    index=False
)

print(
    f"Written: {filtered_library_path}"
)

print(
    f"Written: {retained_bands_path}"
)


Written: /Users/phemelo/Documents/LesHyperion/Data/Hyperspectral/Outputs_324bands/Tables/spectral_library_filtered_324bands.csv
Written: /Users/phemelo/Documents/LesHyperion/Data/Hyperspectral/Outputs_324bands/Tables/retained_bands_324.csv


### 2.2 Diagnostic check of the retained bands

This figure is a visual audit of the wavelength-selection rule. It does not alter the library used by the classification analysis.


In [8]:
# Plot the mean source spectrum and shade the wavelength intervals removed
# by the band-selection rule.

mpl.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 400,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.10,
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

mean_source_reflectance = np.nanmean(
    X_all,
    axis=0
)

fig, ax = plt.subplots(
    figsize=(9.2, 4.0)
)

ax.plot(
    wl_all,
    mean_source_reflectance,
    color="#3F3F3F",
    linewidth=1.0,
)

# Shade the excluded wavelength intervals.

ax.axvspan(
    wl_all.min(),
    DROP_BELOW_NM,
    color="#D9D9D9",
    alpha=0.65,
    linewidth=0,
)

for lower, upper in DROP_RANGES_NM:

    ax.axvspan(
        lower,
        upper,
        color="#D9D9D9",
        alpha=0.65,
        linewidth=0,
    )

ax.axvspan(
    DROP_ABOVE_NM,
    wl_all.max(),
    color="#D9D9D9",
    alpha=0.65,
    linewidth=0,
)

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Mean reflectance")

ax.set_title(
    f"Band selection: {len(band_cols_good)} retained bands",
    loc="left",
    fontweight="bold",
)

fig.tight_layout()

fig.savefig(
    FIG_DIR / "FigS_band_selection_324bands.png"
)

fig.savefig(
    FIG_DIR / "FigS_band_selection_324bands.pdf"
)

plt.show()


## 3. Vector normalisation

The 324-band library created above is now the sole spectral input to the remaining analyses.

Each spectrum is divided by its Euclidean norm. The transformation reduces the influence of overall brightness and places emphasis on spectral shape, which is the basis of the subsequent SAM and classification analyses.


In [9]:
# Extract the retained 324-band reflectance matrix.

spec = (
    filtered_library[band_cols_good]
    .apply(pd.to_numeric, errors="coerce")
)

# Treat the common missing-value code as missing.
spec = spec.replace(-9999, np.nan)


# Remove observations with more than half of their retained bands missing.
#
# This is observation-level quality control. It does not alter the
# definition of the 324 retained wavelengths.

missing_fraction = spec.isna().mean(axis=1)

keep_rows = (
    missing_fraction <= 0.50
)

filtered_library = filtered_library.loc[
    keep_rows
].reset_index(drop=True)

spec = spec.loc[
    keep_rows
].reset_index(drop=True)

print(
    f"Observations before row QC: "
    f"{len(keep_rows):,}"
)

print(
    f"Observations after row QC : "
    f"{keep_rows.sum():,}"
)

print(
    f"Observations removed       : "
    f"{(~keep_rows).sum():,}"
)


Observations before row QC: 1,229
Observations after row QC : 1,229
Observations removed       : 0


In [10]:
# Vector-normalise each retained spectrum.

def vector_normalise(matrix):
    """Divide each spectrum by its Euclidean norm."""

    # Missing values are set to zero after the row-level missingness check.
    matrix = np.nan_to_num(
        matrix,
        nan=0.0
    )

    norms = np.sqrt(
        (matrix ** 2).sum(
            axis=1,
            keepdims=True
        )
    )

    # Avoid division by zero for an entirely empty spectrum.
    norms[norms == 0] = 1.0

    return matrix / norms


X_vn = vector_normalise(
    spec.to_numpy(dtype=float)
)


# The wavelength names remain in wl_good.
# B1...B324 are simply the sequential column names for the normalised matrix.

vn_df = pd.DataFrame(
    X_vn,
    columns=[
        f"B{i + 1}"
        for i in range(X_vn.shape[1])
    ]
)

vn_df.insert(
    0,
    "Species",
    filtered_library[SPECIES_COLUMN].astype(str).values
)

band_cols = [
    column
    for column in vn_df.columns
    if column != "Species"
]

print(
    f"Normalised matrix: "
    f"{X_vn.shape[0]:,} observations × "
    f"{X_vn.shape[1]:,} bands"
)

print(
    f"Value range: "
    f"{X_vn.min():.5f}–{X_vn.max():.5f}"
)


Normalised matrix: 1,229 observations × 324 bands
Value range: -0.00951–0.13122


In [11]:
# Save the final analysis library so the exact input used downstream is
# available outside the notebook.

analysis_library = filtered_library.copy()

analysis_library[
    band_cols_good
] = spec

analysis_library.to_csv(
    TABLE_DIR
    / "spectral_library_analysis_324bands.csv",
    index=False
)

pd.DataFrame({
    "wavelength_nm": wl_good,
    "band_column": band_cols_good,
}).to_csv(
    TABLE_DIR
    / "wavelengths_analysis_324bands.csv",
    index=False
)

print(
    "Final 324-band analysis library written."
)


Final 324-band analysis library written.


## 4. Species-level spectral profiles

These are the manuscript-style spectral profiles. The first panel shows the mean vector-normalised spectrum of every species, ordered by mean near-infrared reflectance. The second panel keeps the same species means but draws *L. argenteum* on top in blue while the other species are shown in grey.

The two grey bands mark the wavelength regions excluded from the analysis. The plotting code uses the same visual grammar as the manuscript figure: coloured species profiles in the first panel, and a strongly highlighted target species in the second.

In [12]:
# Calculate the mean vector-normalised spectrum for each species.

species_mean = (
    vn_df.groupby("Species")[band_cols]
    .mean()
    .reset_index()
)

# Order species by their mean near-infrared reflectance.
# Here NIR is defined as 780–1350 nm, matching the spectral-region analysis.

nir_mask = (wl_good >= 780) & (wl_good <= 1350)
nir_columns = [band_cols[i] for i in np.where(nir_mask)[0]]

species_mean["mean_NIR"] = species_mean[nir_columns].mean(axis=1)
species_mean = species_mean.sort_values("mean_NIR", ascending=False).reset_index(drop=True)

species_order = species_mean["Species"].tolist()

print(f"Species represented in the spectral profiles: {len(species_order)}")

Species represented in the spectral profiles: 26


In [13]:
# Figure 4 — LVIS is deliberately absent here; this is the hyperspectral figure.
# This figure is a direct visualisation of the spectral analysis carried out above.

# It uses:
#         species_mean            -> mean vector-normalised spectrum for each species
#         wl_good                 -> the 324 wavelength bands retained by the integrated band-selection stage
#         TARGET                  -> the species highlighted in the manuscript

# Figure appearance

mpl.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 11,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "#3F3F3F",
})

# The manuscript figure uses a progression from green through blue to purple.
# Viridis gives the same perceptual ordering while remaining readable in print.
profile_colours = plt.get_cmap("viridis")(
    np.linspace(0.20, 0.95, len(species_order))
)

# The target is black in panel (a), matching the attached manuscript figure.
TARGET_BLACK = "#111111"

# The target is blue in panel (b), while the remaining species are grey.
TARGET_BLUE = "#0072B2"
OTHER_GREY = "#A0A0A0"
WATER_GREY = "#D9D9D9"

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10.5, 8.2),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1]}
)

# Panel (a): all species.
ax = axes[0]

for i, species in enumerate(species_order):
    y = species_mean.loc[
        species_mean["Species"] == species,
        band_cols
    ].to_numpy(float).ravel()

    if species == TARGET:
        colour = TARGET_BLACK
        linewidth = 2.8
        zorder = 6
    else:
        colour = profile_colours[i]
        linewidth = 1.4
        zorder = 3

    ax.plot(
        wl_good,
        y,
        color=colour,
        linewidth=linewidth,
        alpha=0.95,
        label=species,
        zorder=zorder,
    )

# Water absorption windows are shown even though the underlying bands were removed.
for lower, upper in [(1340, 1480), (1800, 1980)]:
    ax.axvspan(lower, upper, color=WATER_GREY, alpha=0.55, zorder=0)
    ax.text(
        (lower + upper) / 2,
        0.103,
        "H₂O",
        ha="center",
        va="top",
        fontsize=8,
        color="#777777",
    )

ax.set_ylabel("Vector-normalised reflectance")
ax.set_title(
    "(a) Mean spectrum of each species",
    loc="left",
    fontweight="bold",
)
ax.set_ylim(0, 0.105)

# Keep the long species legend outside the plotting area.
legend = ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    title="Species, ordered by\nnear-infrared reflectance",
    title_fontsize=9,
    frameon=False,
    handlelength=2.0,
)

# Italicise scientific names without changing the species labels themselves.
for text in legend.get_texts():
    text.set_fontstyle("italic")

# Panel (b): the same means, with the target species highlighted.
ax = axes[1]

for species in species_order:
    if species == TARGET:
        continue

    y = species_mean.loc[
        species_mean["Species"] == species,
        band_cols
    ].to_numpy(float).ravel()

    ax.plot(
        wl_good,
        y,
        color=OTHER_GREY,
        linewidth=1.0,
        alpha=0.55,
        zorder=2,
    )

# Draw the target last so that it remains visible over overlapping spectra.
target_y = species_mean.loc[
    species_mean["Species"] == TARGET,
    band_cols
].to_numpy(float).ravel()

target_n = int((vn_df["Species"] == TARGET).sum())

# Plot the individual target spectra faintly behind the mean where possible.
target_individual = vn_df.loc[
    vn_df["Species"] == TARGET,
    band_cols
].to_numpy(float)

if len(target_individual) > 0:
    target_sd = target_individual.std(axis=0, ddof=1)

    ax.fill_between(
        wl_good,
        target_y - target_sd,
        target_y + target_sd,
        color=TARGET_BLUE,
        alpha=0.12,
        linewidth=0,
        zorder=3,
    )

ax.plot(
    wl_good,
    target_y,
    color=TARGET_BLUE,
    linewidth=2.8,
    label=f"L. argenteum (n = {target_n})",
    zorder=5,
)

for lower, upper in [(1340, 1480), (1800, 1980)]:
    ax.axvspan(lower, upper, color=WATER_GREY, alpha=0.55, zorder=0)
    ax.text(
        (lower + upper) / 2,
        0.103,
        "H₂O",
        ha="center",
        va="top",
        fontsize=8,
        color="#777777",
    )

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Vector-normalised reflectance")
ax.set_title(
    "(b) The same means, with $L.$ $argenteum$ highlighted",
    loc="left",
    fontweight="bold",
)
ax.set_xlim(wl_good.min(), wl_good.max())
ax.set_ylim(0, 0.105)

legend = ax.legend(
    frameon=False,
    loc="upper right",
)
legend.get_texts()[0].set_fontstyle("italic")

fig.tight_layout()

fig.savefig(FIG_DIR / "Fig4_spectral_profiles.png")
fig.savefig(FIG_DIR / "Fig4_spectral_profiles.pdf")

plt.show()

print(f"Figure 4 saved with {len(wl_good)} wavelength bands and {len(species_order)} species.")


Figure 4 saved with 324 wavelength bands and 26 species.


## 5. Spectral Angle Mapper

Spectral Angle Mapper (SAM) treats spectra as vectors and measures the angle between them. A smaller angle means two spectral shapes are more similar. The analysis below first calculates the complete species-by-species angle matrix and then performs permutation tests for *L. argenteum* against every other species.

The permutation test is used here as a test of **mean-spectrum separation**. It is not interpreted as a test of classifier accuracy.

In [14]:
# Calculate one mean spectrum per species and convert the means to unit vectors.

species_names = species_order.copy()

species_vectors = []
for species in species_names:
    mean_vector = species_mean.loc[
        species_mean["Species"] == species,
        band_cols
    ].to_numpy(float).ravel()

    norm = np.linalg.norm(mean_vector)
    species_vectors.append(mean_vector / norm if norm > 0 else mean_vector)

species_vectors = np.vstack(species_vectors)

# The dot product between unit vectors is their cosine similarity.
cosine_matrix = np.clip(
    species_vectors @ species_vectors.T,
    -1.0,
    1.0,
)

angle_matrix = np.degrees(np.arccos(cosine_matrix))

sam_df = pd.DataFrame(
    angle_matrix,
    index=species_names,
    columns=species_names,
)

sam_df.to_csv(TABLE_DIR / "SAM_species_angle_matrix_degrees.csv")

# Find the species whose mean spectra are most similar to L. argenteum.
target_angles = sam_df.loc[TARGET].drop(TARGET).sort_values()

print(f"Nearest spectral neighbours of {TARGET}:")
print(target_angles.head(10).round(3).to_string())

Nearest spectral neighbours of Leucadendron argenteum:
Pteridium aquilinum capense               2.083
Leucospermum conocarpodendron viridum     8.243
Mimetes hirtus                            9.625
Psoralea fruticans                        9.873
Drosera aliciae                          10.476
Liparia splendens splendens              10.834
Pelargonium cucullatum tabulare          10.904
Protea lepidocarpodendron                12.186
Gladiolus carneus                        12.632
Disa rosea                               13.433


In [15]:
# Permutation tests for the target species.

rng = np.random.default_rng(RANDOM_STATE)

sam_perm_rows = []

X_target = vn_df.loc[vn_df["Species"] == TARGET, band_cols].to_numpy(float)

for species in target_angles.index:
    X_other = vn_df.loc[vn_df["Species"] == species, band_cols].to_numpy(float)
    pooled = np.vstack([X_target, X_other])
    n_target = len(X_target)
    observed = float(sam_df.loc[TARGET, species])

    null_angles = np.empty(SAM_PERMUTATIONS, dtype=float)

    for permutation in range(SAM_PERMUTATIONS):
        order = rng.permutation(len(pooled))

        group_a = pooled[order[:n_target]].mean(axis=0)
        group_b = pooled[order[n_target:]].mean(axis=0)

        norm_a = np.linalg.norm(group_a)
        norm_b = np.linalg.norm(group_b)

        if norm_a == 0 or norm_b == 0:
            null_angles[permutation] = np.nan
            continue

        group_a = group_a / norm_a
        group_b = group_b / norm_b

        null_angles[permutation] = np.degrees(
            np.arccos(np.clip(np.dot(group_a, group_b), -1.0, 1.0))
        )

    valid_null = null_angles[np.isfinite(null_angles)]
    p_value = (np.sum(valid_null >= observed) + 1) / (len(valid_null) + 1)

    sam_perm_rows.append({
        "Species": species,
        "SAM_deg": observed,
        "p_perm": p_value,
    })

sam_perm = pd.DataFrame(sam_perm_rows).sort_values("SAM_deg").reset_index(drop=True)

# Holm correction keeps the family-wise comparison focused on the target-vs-rest tests.
ordered_p = sam_perm["p_perm"].to_numpy()
multiplier = len(ordered_p) - np.arange(len(ordered_p))
sam_perm["p_holm"] = np.minimum(1.0, ordered_p * multiplier)

sam_perm.to_csv(TABLE_DIR / "SAM_target_permutation_tests.csv", index=False)

print("Target-versus-rest SAM permutation results:")
print(sam_perm.head(10).round(4).to_string(index=False))

Target-versus-rest SAM permutation results:
                              Species  SAM_deg  p_perm  p_holm
          Pteridium aquilinum capense   2.0827   0.229   1.000
Leucospermum conocarpodendron viridum   8.2429   0.001   0.024
                       Mimetes hirtus   9.6252   0.001   0.023
                   Psoralea fruticans   9.8733   0.001   0.022
                      Drosera aliciae  10.4757   0.001   0.021
          Liparia splendens splendens  10.8335   0.001   0.020
      Pelargonium cucullatum tabulare  10.9036   0.001   0.019
            Protea lepidocarpodendron  12.1863   0.001   0.018
                    Gladiolus carneus  12.6318   0.001   0.017
                           Disa rosea  13.4327   0.001   0.016


In [16]:
# Draw the full SAM matrix.

# Keep only one triangle so each species pair appears once.
mask_upper = np.triu(np.ones_like(angle_matrix, dtype=bool), k=1)
masked_angles = np.ma.array(angle_matrix, mask=mask_upper | np.eye(len(species_names), dtype=bool))

fig, ax = plt.subplots(
    figsize=(10, 9)
)

image = ax.imshow(
    masked_angles,
    cmap="viridis",
    aspect="equal",
)

ax.set_xticks(range(len(species_names)))
ax.set_yticks(range(len(species_names)))
ax.set_xticklabels(species_names, rotation=90, fontsize=7)
ax.set_yticklabels(species_names, fontsize=7)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontstyle("italic")

for i in range(len(species_names)):
    for j in range(len(species_names)):
        if mask_upper[i, j] or i == j:
            continue

        ax.text(
            j,
            i,
            f"{angle_matrix[i, j]:.1f}°",
            ha="center",
            va="center",
            fontsize=5.6,
            color="white",
        )

        ax.add_patch(
            Rectangle(
                (j - 0.5, i - 0.5),
                1,
                1,
                fill=False,
                edgecolor="white",
                linewidth=0.35,
            )
        )

cbar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Spectral angle (degrees)")

ax.set_title(
    "Inter-species spectral angle matrix — SAM",
    loc="left",
    fontweight="bold",
    pad=10,
)

fig.tight_layout()
fig.savefig(FIG_DIR / "FigS_SAM_matrix.png")
fig.savefig(FIG_DIR / "FigS_SAM_matrix.pdf")
plt.show()

## 6. PCA dimensionality reduction

PCA is used to reduce the 324 correlated wavelength bands to a smaller set of orthogonal components before classification. The number of components is fixed at 10 to match the original modelling workflow.

The fit shown here is useful for visualisation. The classification models below fit their PCA transformation **inside each cross-validation training fold**, which prevents information from the held-out observations entering the PCA solution.

In [17]:
# Prepare the classification dataset.

counts = vn_df["Species"].value_counts()
eligible_species = counts[counts >= MIN_SAMPLES].index.tolist()

df_cv = vn_df[vn_df["Species"].isin(eligible_species)].reset_index(drop=True)
X_class = df_cv[band_cols].to_numpy(float)
y_species = df_cv["Species"].to_numpy()

print(f"Eligible species : {len(eligible_species)}")
print(f"Total spectra    : {len(df_cv):,}")
print("\nSpecies sample sizes:")
print(df_cv["Species"].value_counts().sort_index().to_string())

if TARGET not in eligible_species:
    raise ValueError(f"{TARGET} does not meet the minimum sample requirement.")

Eligible species : 26
Total spectra    : 1,229

Species sample sizes:
Species
Audouinia capitata                       23
Disa rosea                               48
Drosera aliciae                          50
Drosera cuneifolia                       50
Erica corifolia                          50
Erica pyxidiflora                        30
Gladiolus carneus                        48
Hermas villosa                           50
Leucadendron argenteum                   50
Leucadendron xanthoconus                 50
Leucospermum conocarpodendron viridum    50
Liparia splendens splendens              50
Mimetes fimbriifolius                    50
Mimetes hirtus                           49
Pelargonium cucullatum tabulare          48
Protea cynaroides                        50
Protea lepidocarpodendron                50
Protea speciosa                          50
Psoralea fruticans                       49
Pteridium aquilinum capense              50
Roella ciliata                           4

In [18]:
# Fit PCA once for the descriptive PCA figure.

n_pcs = min(N_PCS, X_class.shape[0] - 1, X_class.shape[1] - 1)

pca_visual = PCA(
    n_components=n_pcs,
    random_state=RANDOM_STATE,
)

Z_visual = pca_visual.fit_transform(X_class)
variance_explained = pca_visual.explained_variance_ratio_

print(f"PCA components retained : {n_pcs}")
print(f"Cumulative variance     : {variance_explained.sum() * 100:.1f}%")

PCA components retained : 10
Cumulative variance     : 99.6%


In [19]:
# Figure — PCA scree plot and PC1/PC2 scores.

# Build a consistent colour map for the species.
pca_colours = {
    species: plt.get_cmap("viridis")(
        0.20 + 0.75 * i / max(len(eligible_species) - 1, 1)
    )
    for i, species in enumerate(sorted(eligible_species))
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.7),
    gridspec_kw={"width_ratios": [0.9, 1.1]},
)

# Scree plot.
ax = axes[0]
components = np.arange(1, n_pcs + 1)

ax.bar(
    components,
    variance_explained * 100,
    color="#56B4E9",
    edgecolor="white",
    linewidth=0.5,
)

ax.plot(
    components,
    variance_explained.cumsum() * 100,
    marker="o",
    color="#0072B2",
    linewidth=1.5,
)

ax.set_xlabel("Principal component")
ax.set_ylabel("Variance explained (%)")
ax.set_title("(a) PCA variance explained", loc="left", fontweight="bold")

# PC1 versus PC2.
ax = axes[1]

for species in sorted(eligible_species):
    selected = y_species == species

    ax.scatter(
        Z_visual[selected, 0],
        Z_visual[selected, 1],
        s=22 if species != TARGET else 40,
        color=pca_colours[species],
        edgecolor="#111111" if species == TARGET else "none",
        linewidth=1.1 if species == TARGET else 0,
        alpha=0.80,
        label=species,
    )

ax.set_xlabel(f"PC1 ({variance_explained[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({variance_explained[1] * 100:.1f}%)")
ax.set_title("(b) PCA scores", loc="left", fontweight="bold")

legend = ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=7,
)

for text in legend.get_texts():
    text.set_fontstyle("italic")

fig.tight_layout()
fig.savefig(FIG_DIR / "FigS_PCA.png")
fig.savefig(FIG_DIR / "FigS_PCA.pdf")
plt.show()

## 7. Classification models and leakage-free cross-validation

Six classifiers are compared:

- logistic regression;
- linear support vector machine;
- radial-basis-function support vector machine;
- random forest;
- k-nearest neighbours; and
- XGBoost.

Each model receives the same 5-fold stratified splits. PCA is fitted separately inside each training fold rather than once on the full dataset. This distinction matters because a PCA transformation estimated from the full dataset would allow the test observations to influence the predictor space.

In [20]:
# Create the common classifier definitions.

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_species)
target_encoded = int(label_encoder.transform([TARGET])[0])

min_class_size = int(counts[eligible_species].min())
if min_class_size < CV_FOLDS:
    raise ValueError(
        f"At least {CV_FOLDS} observations per class are required for the requested "
        f"5-fold CV. The smallest eligible class has {min_class_size}."
    )

skf = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

k_neighbors = min(5, min_class_size - 1)

classifiers = {
    "Logistic regression": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        C=1.0,
    ),
    "SVM (linear)": SVC(
        kernel="linear",
        probability=True,
        random_state=RANDOM_STATE,
    ),
    "SVM (RBF)": SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_STATE,
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=RF_TREES,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "k-NN": KNeighborsClassifier(
        n_neighbors=k_neighbors,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    ),
}

print("Classifiers:")
for name in classifiers:
    print(f"  {name}")

Classifiers:
  Logistic regression
  SVM (linear)
  SVM (RBF)
  Random forest
  k-NN
  XGBoost


In [21]:
# Build the same PCA pipeline for every classifier.

# PCA is fitted within each training fold by the Pipeline object.
# No PCA object is fitted on the entire dataset for the performance analysis.

def make_pipeline(classifier):
    return Pipeline([
        ("pca", PCA(
            n_components=n_pcs,
            random_state=RANDOM_STATE,
        )),
        ("classifier", classifier),
    ])

print("PCA is part of every cross-validation pipeline.")

PCA is part of every cross-validation pipeline.


In [22]:
# Run the six classifiers using the same folds.

# Store out-of-fold predictions so that every performance metric is based on
# observations that were genuinely held out from the corresponding fit.

results_by_model = {}
oof_predictions = {}

labels = np.arange(len(label_encoder.classes_))

for model_name, estimator in classifiers.items():
    print(f"Running {model_name} ...")

    oof_pred = np.full(len(y_encoded), -1, dtype=int)
    oof_prob = np.full(len(y_encoded), np.nan, dtype=float)

    for train_idx, test_idx in skf.split(X_class, y_encoded):
        pipeline = make_pipeline(estimator)

        pipeline.fit(
            X_class[train_idx],
            y_encoded[train_idx],
        )

        predicted = pipeline.predict(X_class[test_idx]).astype(int)
        probability = pipeline.predict_proba(X_class[test_idx])

        oof_pred[test_idx] = predicted

        # Find the position of the target class in the model's probability matrix.
        classes_seen = pipeline.named_steps["classifier"].classes_
        target_position = int(np.where(classes_seen == target_encoded)[0][0])
        oof_prob[test_idx] = probability[:, target_position]

    results_by_model[model_name] = {
        "prediction": oof_pred,
        "probability_target": oof_prob,
    }

    oof_predictions[model_name] = pd.DataFrame({
        "y_true": y_encoded,
        "y_pred": oof_pred,
        "p_target": oof_prob,
        "Species_true": label_encoder.inverse_transform(y_encoded),
        "Species_pred": label_encoder.inverse_transform(oof_pred),
    })

print("\nAll classifiers completed.")

Running Logistic regression ...
Running SVM (linear) ...
Running SVM (RBF) ...
Running Random forest ...
Running k-NN ...
Running XGBoost ...

All classifiers completed.


In [23]:
# Calculate the performance metrics from the out-of-fold predictions.

metric_rows = []

for model_name, output in results_by_model.items():
    y_pred = output["prediction"]

    metric_rows.append({
        "Model": model_name,
        "Overall accuracy": accuracy_score(y_encoded, y_pred),
        "Target sensitivity": recall_score(
            y_encoded,
            y_pred,
            labels=[target_encoded],
            average="micro",
            zero_division=0,
        ),
        "Target precision": precision_score(
            y_encoded,
            y_pred,
            labels=[target_encoded],
            average="micro",
            zero_division=0,
        ),
        "Target F1": f1_score(
            y_encoded,
            y_pred,
            labels=[target_encoded],
            average="micro",
            zero_division=0,
        ),
    })

metrics_df = pd.DataFrame(metric_rows).sort_values(
    "Target F1",
    ascending=False,
).reset_index(drop=True)

metrics_df.to_csv(TABLE_DIR / "classifier_performance_324bands.csv", index=False)

print(metrics_df.round(4).to_string(index=False))

              Model  Overall accuracy  Target sensitivity  Target precision  Target F1
            XGBoost            0.2815                0.70            0.5469     0.6140
      Random forest            0.3059                0.76            0.5000     0.6032
               k-NN            0.1733                0.64            0.3765     0.4741
          SVM (RBF)            0.1717                0.78            0.2708     0.4021
       SVM (linear)            0.0862                0.88            0.1705     0.2857
Logistic regression            0.1033                0.94            0.1536     0.2640


## 8. Target-species performance and bootstrap confidence intervals

The classifier comparison above identifies the best model by the *L. argenteum* F1-score. The bootstrap below resamples the complete set of out-of-fold predictions of that model 1,000 times. The confidence interval therefore describes uncertainty in the observed out-of-fold performance, rather than pretending that a bootstrap fit is an independent validation dataset.

In [24]:
# Select the model with the highest target-species F1-score.

BEST_MODEL = metrics_df.loc[0, "Model"]
BEST_OUTPUT = results_by_model[BEST_MODEL]
BEST_PREDICTIONS = oof_predictions[BEST_MODEL].copy()

print(f"Best classifier by target F1: {BEST_MODEL}")
print(f"Target F1: {metrics_df.loc[0, 'Target F1']:.4f}")

Best classifier by target F1: XGBoost
Target F1: 0.6140


In [25]:
# Bootstrap target F1 and sensitivity from the out-of-fold predictions.

rng = np.random.default_rng(RANDOM_STATE)

observed_true = y_encoded.copy()
observed_pred = BEST_OUTPUT["prediction"].copy()

bootstrap_f1 = np.empty(BOOTSTRAP_REPS)
bootstrap_sensitivity = np.empty(BOOTSTRAP_REPS)
bootstrap_accuracy = np.empty(BOOTSTRAP_REPS)

n_observations = len(observed_true)

for repetition in range(BOOTSTRAP_REPS):
    indices = rng.integers(
        low=0,
        high=n_observations,
        size=n_observations,
    )

    y_true_boot = observed_true[indices]
    y_pred_boot = observed_pred[indices]

    bootstrap_f1[repetition] = f1_score(
        y_true_boot,
        y_pred_boot,
        labels=[target_encoded],
        average="micro",
        zero_division=0,
    )

    bootstrap_sensitivity[repetition] = recall_score(
        y_true_boot,
        y_pred_boot,
        labels=[target_encoded],
        average="micro",
        zero_division=0,
    )

    bootstrap_accuracy[repetition] = accuracy_score(
        y_true_boot,
        y_pred_boot,
    )


def percentile_ci(values):
    return np.percentile(values, [2.5, 97.5])

f1_ci = percentile_ci(bootstrap_f1)
sensitivity_ci = percentile_ci(bootstrap_sensitivity)
accuracy_ci = percentile_ci(bootstrap_accuracy)

bootstrap_summary = pd.DataFrame({
    "Metric": [
        "Target F1",
        "Target sensitivity",
        "Overall accuracy",
    ],
    "Observed": [
        f1_score(observed_true, observed_pred, labels=[target_encoded], average="micro", zero_division=0),
        recall_score(observed_true, observed_pred, labels=[target_encoded], average="micro", zero_division=0),
        accuracy_score(observed_true, observed_pred),
    ],
    "CI_2.5": [f1_ci[0], sensitivity_ci[0], accuracy_ci[0]],
    "CI_97.5": [f1_ci[1], sensitivity_ci[1], accuracy_ci[1]],
})

bootstrap_summary.to_csv(
    TABLE_DIR / "bootstrap_confidence_intervals_best_classifier.csv",
    index=False,
)

print(bootstrap_summary.round(4).to_string(index=False))

            Metric  Observed  CI_2.5  CI_97.5
         Target F1    0.6140  0.5000   0.7119
Target sensitivity    0.7000  0.5600   0.8246
  Overall accuracy    0.2815  0.2579   0.3076


In [26]:
# Draw the bootstrap distributions for the best classifier.

fig, axes = plt.subplots(
    1,
    3,
    figsize=(11.5, 3.8),
)

bootstrap_sets = [
    (bootstrap_f1, "Target F1", f1_ci),
    (bootstrap_sensitivity, "Target sensitivity", sensitivity_ci),
    (bootstrap_accuracy, "Overall accuracy", accuracy_ci),
]

for ax, (values, label, ci) in zip(axes, bootstrap_sets):
    ax.hist(
        values,
        bins=30,
        color="#56B4E9",
        edgecolor="white",
        linewidth=0.5,
    )

    observed = float(bootstrap_summary.loc[
        bootstrap_summary["Metric"] == label,
        "Observed"
    ].iloc[0])

    ax.axvline(
        observed,
        color="#D55E00",
        linewidth=1.6,
        label="Observed",
    )

    ax.axvline(ci[0], color="#4D4D4D", linestyle="--", linewidth=0.9)
    ax.axvline(ci[1], color="#4D4D4D", linestyle="--", linewidth=0.9)

    ax.set_xlabel(label)
    ax.set_ylabel("Bootstrap replicates")

fig.suptitle(
    f"Bootstrap uncertainty for {BEST_MODEL}",
    fontsize=11,
    fontweight="bold",
)

fig.tight_layout()
fig.savefig(FIG_DIR / "FigS_bootstrap_best_classifier.png")
fig.savefig(FIG_DIR / "FigS_bootstrap_best_classifier.pdf")
plt.show()

## 9. Per-species performance and confusion structure

The manuscript evaluates the target species within the complete multi-species classification problem. The table and figure below therefore use the out-of-fold predictions from the best classifier and calculate precision, sensitivity and F1 independently for every species.

The confusion analysis then distinguishes two different phenomena: true *L. argenteum* observations that were predicted as another species, and observations of other species that were predicted as *L. argenteum*. This makes the source of false negatives and false positives explicit.

In [27]:
# Calculate per-species metrics from the best classifier's out-of-fold predictions.

per_species_rows = []

for encoded_species, species in enumerate(label_encoder.classes_):
    true_mask = y_encoded == encoded_species
    pred_mask = observed_pred == encoded_species

    tp = int(np.sum(true_mask & pred_mask))
    fn = int(np.sum(true_mask & ~pred_mask))
    fp = int(np.sum(~true_mask & pred_mask))

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (
        2 * precision * sensitivity / (precision + sensitivity)
        if (precision + sensitivity)
        else 0.0
    )

    per_species_rows.append({
        "Species": species,
        "n": int(true_mask.sum()),
        "Sensitivity": sensitivity,
        "Precision": precision,
        "F1": f1,
    })

per_species = pd.DataFrame(per_species_rows).sort_values(
    "F1",
    ascending=False,
).reset_index(drop=True)

per_species["Rank"] = np.arange(1, len(per_species) + 1)

per_species.to_csv(
    TABLE_DIR / "per_species_performance_best_classifier_324bands.csv",
    index=False,
)

target_rank = int(
    per_species.loc[per_species["Species"] == TARGET, "Rank"].iloc[0]
)

target_f1 = float(
    per_species.loc[per_species["Species"] == TARGET, "F1"].iloc[0]
)

print(
    f"{TARGET}: rank {target_rank} of {len(per_species)} by F1 "
    f"(F1 = {target_f1:.3f})"
)

print("\nTop species by F1:")
print(per_species.head(10).round(3).to_string(index=False))

Leucadendron argenteum: rank 2 of 26 by F1 (F1 = 0.614)

Top species by F1:
                    Species  n  Sensitivity  Precision    F1  Rank
             Witsenia maura 46        0.826      0.704 0.760     1
     Leucadendron argenteum 50        0.700      0.547 0.614     2
          Erica pyxidiflora 30        0.667      0.500 0.571     3
         Serruria glomerata 45        0.556      0.490 0.521     4
         Audouinia capitata 23        0.391      0.600 0.474     5
             Mimetes hirtus 49        0.469      0.460 0.465     6
           Serruria villosa 49        0.469      0.371 0.414     7
         Drosera cuneifolia 50        0.420      0.328 0.368     8
Pteridium aquilinum capense 50        0.320      0.302 0.311     9
                 Disa rosea 48        0.312      0.273 0.291    10


In [28]:
# Identify the species that are most often involved in target confusion.

oof = BEST_PREDICTIONS.copy()

oof["true_species"] = label_encoder.inverse_transform(y_encoded)
oof["pred_species"] = label_encoder.inverse_transform(observed_pred)

target_true = oof[oof["true_species"] == TARGET]
target_pred = oof[oof["pred_species"] == TARGET]

# False negatives: true target observations predicted as something else.
missed = (
    target_true[target_true["pred_species"] != TARGET]["pred_species"]
    .value_counts()
    .rename("n")
    .to_frame()
)

if len(missed):
    missed["pct_target_missed"] = missed["n"] / len(target_true) * 100
    missed["SAM_deg_to_target"] = [sam_df.loc[TARGET, species] for species in missed.index]

# False positives: other species predicted as the target.
false_positive = (
    target_pred[target_pred["true_species"] != TARGET]["true_species"]
    .value_counts()
    .rename("n")
    .to_frame()
)

if len(false_positive):
    false_positive["pct_target_predictions"] = (
        false_positive["n"] / len(target_pred) * 100
    )
    false_positive["SAM_deg_to_target"] = [
        sam_df.loc[TARGET, species]
        for species in false_positive.index
    ]

missed.to_csv(TABLE_DIR / "target_false_negatives_by_species.csv")
false_positive.to_csv(TABLE_DIR / "target_false_positives_by_species.csv")

print("Most common false-positive species:")
print(false_positive.head(8).round(3).to_string())

print("\nMost common false-negative species:")
print(missed.head(8).round(3).to_string())

Most common false-positive species:
                                       n  pct_target_predictions  SAM_deg_to_target
true_species                                                                       
Pelargonium cucullatum tabulare        6                   9.375             10.904
Gladiolus carneus                      5                   7.812             12.632
Pteridium aquilinum capense            5                   7.812              2.083
Roella ciliata                         3                   4.688             14.450
Leucospermum conocarpodendron viridum  3                   4.688              8.243
Leucadendron xanthoconus               2                   3.125             14.534
Psoralea fruticans                     2                   3.125              9.873
Drosera aliciae                        1                   1.562             10.476

Most common false-negative species:
                                       n  pct_target_missed  SAM_deg_to_target
pred_spe

In [29]:
# Figure 5 — classification performance and target confusion.

# Panel (a) ranks species by their F1-score.
# Panel (b) shows the species most often predicted as L. argenteum and
# places their spectral angle to the target on the same plot.

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 5.4),
    gridspec_kw={"width_ratios": [1.0, 0.9]},
)

# Panel (a): per-species F1 ranking.
ax = axes[0]
show = per_species.iloc[::-1].copy()

bar_colours = [
    TARGET_BLUE if species == TARGET else "#9FB6C9"
    for species in show["Species"]
]

labels_plot = [
    f"{species} ({n})"
    for species, n in zip(show["Species"], show["n"])
]

ax.barh(
    labels_plot,
    show["F1"],
    color=bar_colours,
    height=0.72,
)

ax.axvline(
    per_species["F1"].median(),
    color="#777777",
    linestyle=":",
    linewidth=1.0,
)

ax.set_xlabel("F1-score")
ax.set_title(
    "(a) Per-species F1-score",
    loc="left",
    fontweight="bold",
)

# Panel (b): false positives and their SAM similarity.
ax = axes[1]

if len(false_positive):
    top_fp = false_positive.head(8).copy().iloc[::-1]

    y_positions = np.arange(len(top_fp))

    ax.barh(
        y_positions,
        top_fp["n"],
        color="#56B4E9",
        height=0.68,
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(top_fp.index)

    for text in ax.get_yticklabels():
        text.set_fontstyle("italic")

    ax.set_xlabel("False positives predicted as L. argenteum")

    ax2 = ax.twiny()
    ax2.scatter(
        top_fp["SAM_deg_to_target"],
        y_positions,
        color="#D55E00",
        s=28,
        zorder=5,
    )
    ax2.set_xlabel("Spectral angle to L. argenteum (degrees)")
    ax2.set_xlim(left=0)

else:
    ax.text(
        0.5,
        0.5,
        "No false-positive observations",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

ax.set_title(
    "(b) Species predicted as L. argenteum",
    loc="left",
    fontweight="bold",
)

fig.tight_layout()
fig.savefig(FIG_DIR / "Fig5_classification_performance.png")
fig.savefig(FIG_DIR / "Fig5_classification_performance.pdf")
plt.show()

## 10. Spectral basis of discrimination

This section asks why *L. argenteum* can be separated despite substantial overlap between species. Three analyses are used together:

1. spectra of the target and its nearest/actual confusers;
2. leave-one-region-out ablation of the classifier; and
3. SHAP attribution for XGBoost, with the contribution mapped back towards wavelength space.

The first analysis describes the observed spectral differences directly. The ablation analysis asks whether removing broad wavelength regions reduces target F1. SHAP addresses a different question: which predictor dimensions the fitted XGBoost model uses most strongly.

In [30]:
# Select a small set of confusers using three pieces of evidence:
# false positives, false negatives, and nearest SAM neighbours.

candidate_confusers = []

for species in false_positive.index.tolist():
    if species != TARGET and species not in candidate_confusers:
        candidate_confusers.append(species)

for species in missed.index.tolist():
    if species != TARGET and species not in candidate_confusers:
        candidate_confusers.append(species)

for species in target_angles.index.tolist():
    if species != TARGET and species not in candidate_confusers:
        candidate_confusers.append(species)

confusers = candidate_confusers[:3]

print("Confusers carried into the spectral-basis figure:")
for species in confusers:
    print(f"  {species}")

Confusers carried into the spectral-basis figure:
  Pelargonium cucullatum tabulare
  Gladiolus carneus
  Pteridium aquilinum capense


In [33]:
# ============================================================================
# Leave-one-region-out spectral ablation
# Repeated cross-validation + confidence intervals + permutation tests
# ============================================================================

import itertools
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

from statsmodels.stats.multitest import multipletests


# ----------------------------------------------------------------------------
# Analysis settings
# ----------------------------------------------------------------------------

ABLATION_REPEATS = 10
CV_FOLDS = 5
BOOTSTRAP_REPS = 10000
RANDOM_STATE = 42


SPECTRAL_REGIONS = {
    "Visible": (450, 700),
    "Red-edge": (700, 780),
    "NIR": (780, 1340),
    "SWIR-1": (1480, 1800),
    "SWIR-2": (1980, 2400),
}


# ----------------------------------------------------------------------------
# Build repeated stratified folds
#
# The same train/test partitions are used for the full-spectrum model and
# every ablated model. This makes the comparisons paired.
# ----------------------------------------------------------------------------

repeat_folds = []

for repeat in range(ABLATION_REPEATS):

    splitter = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE + repeat,
    )

    repeat_folds.append(
        list(
            splitter.split(
                X_class,
                y_encoded,
            )
        )
    )


print(
    f"{ABLATION_REPEATS} repeats × {CV_FOLDS} folds "
    f"= {ABLATION_REPEATS * CV_FOLDS} paired folds"
)


# ----------------------------------------------------------------------------
# Select the classifier identified as best in the main classification analysis
# ----------------------------------------------------------------------------

best_estimator = classifiers[BEST_MODEL]

print(f"Classifier used: {BEST_MODEL}")


# ----------------------------------------------------------------------------
# FULL-SPECTRUM BASELINE
# ----------------------------------------------------------------------------

full_scores = []


print("\nRunning full-spectrum baseline...")


for repeat_number, folds in enumerate(repeat_folds, start=1):

    for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):

        model = clone(best_estimator)

        fold_pipeline = Pipeline([
            (
                "pca",
                PCA(
                    n_components=N_PCS,
                    random_state=RANDOM_STATE,
                ),
            ),
            (
                "classifier",
                model,
            ),
        ])


        fold_pipeline.fit(
            X_class[train_idx],
            y_encoded[train_idx],
        )


        prediction = fold_pipeline.predict(
            X_class[test_idx]
        ).astype(int)


        fold_f1 = f1_score(
            y_encoded[test_idx],
            prediction,
            labels=[target_encoded],
            average="micro",
            zero_division=0,
        )


        full_scores.append(fold_f1)


full_scores = np.asarray(full_scores)


# Convert the 50 fold scores to 10 repeat-level mean F1 values.

full_scores_by_repeat = full_scores.reshape(
    ABLATION_REPEATS,
    CV_FOLDS,
).mean(axis=1)


print(
    f"Full-spectrum mean F1 = {full_scores.mean():.4f}"
)


# ----------------------------------------------------------------------------
# LEAVE-ONE-REGION-OUT ANALYSIS
# ----------------------------------------------------------------------------

ablation_statistics = []

ablation_fold_scores = {}


for region, (lower, upper) in SPECTRAL_REGIONS.items():

    print(
        f"\nRemoving {region}: "
        f"{lower}–{upper} nm"
    )


    # Identify retained wavelengths outside the region being removed.

    keep_mask = ~(
        (wl_good >= lower)
        & (wl_good < upper)
    )


    bands_removed = int(
        (~keep_mask).sum()
    )


    bands_remaining = int(
        keep_mask.sum()
    )


    print(
        f"  Bands removed   : {bands_removed}"
    )

    print(
        f"  Bands remaining : {bands_remaining}"
    )


    if bands_removed == 0:

        print(
            f"  Skipping {region}: "
            "no retained wavelengths occur in this region."
        )

        continue


    if bands_remaining <= N_PCS:

        print(
            f"  Skipping {region}: "
            f"fewer than {N_PCS} bands would remain."
        )

        continue


    reduced_X = X_class[:, keep_mask]


    reduced_scores = []


    # ------------------------------------------------------------------------
    # Run exactly the same repeated CV partitions as the full-spectrum model
    # ------------------------------------------------------------------------

    for repeat_number, folds in enumerate(repeat_folds, start=1):

        for fold_number, (train_idx, test_idx) in enumerate(
            folds,
            start=1,
        ):

            model = clone(best_estimator)


            fold_pipeline = Pipeline([
                (
                    "pca",
                    PCA(
                        n_components=N_PCS,
                        random_state=RANDOM_STATE,
                    ),
                ),
                (
                    "classifier",
                    model,
                ),
            ])


            fold_pipeline.fit(
                reduced_X[train_idx],
                y_encoded[train_idx],
            )


            prediction = fold_pipeline.predict(
                reduced_X[test_idx]
            ).astype(int)


            fold_f1 = f1_score(
                y_encoded[test_idx],
                prediction,
                labels=[target_encoded],
                average="micro",
                zero_division=0,
            )


            reduced_scores.append(
                fold_f1
            )


    reduced_scores = np.asarray(
        reduced_scores
    )


    ablation_fold_scores[region] = reduced_scores


    # ------------------------------------------------------------------------
    # Aggregate the five folds within each repeat
    #
    # This gives 10 paired repeat-level estimates rather than treating all
    # 50 CV folds as statistically independent observations.
    # ------------------------------------------------------------------------

    reduced_scores_by_repeat = reduced_scores.reshape(
        ABLATION_REPEATS,
        CV_FOLDS,
    ).mean(axis=1)


    delta_by_repeat = (
        reduced_scores_by_repeat
        - full_scores_by_repeat
    )


    mean_full_f1 = full_scores_by_repeat.mean()

    mean_reduced_f1 = reduced_scores_by_repeat.mean()

    mean_delta_f1 = delta_by_repeat.mean()


    # ------------------------------------------------------------------------
    # Bootstrap 95% confidence interval for mean ΔF1
    #
    # Resample the 10 repeat-level paired differences.
    # ------------------------------------------------------------------------

    random_generator = np.random.default_rng(
        RANDOM_STATE
    )


    bootstrap_delta = []


    for bootstrap_iteration in range(
        BOOTSTRAP_REPS
    ):

        bootstrap_indices = random_generator.choice(
            ABLATION_REPEATS,
            size=ABLATION_REPEATS,
            replace=True,
        )


        bootstrap_delta.append(
            delta_by_repeat[
                bootstrap_indices
            ].mean()
        )


    bootstrap_delta = np.asarray(
        bootstrap_delta
    )


    confidence_interval_lower = np.percentile(
        bootstrap_delta,
        2.5,
    )

    confidence_interval_upper = np.percentile(
        bootstrap_delta,
        97.5,
    )


    # ------------------------------------------------------------------------
    # Exact paired permutation test
    #
    # Under the null hypothesis, the direction of each repeat-level paired
    # difference is arbitrary. With 10 repeats there are only 2^10 = 1,024
    # possible sign combinations, so the permutation test can be exact.
    # ------------------------------------------------------------------------

    observed_difference = abs(
        mean_delta_f1
    )


    permuted_means = []


    for signs in itertools.product(
        [-1, 1],
        repeat=ABLATION_REPEATS,
    ):

        signs = np.asarray(
            signs
        )


        permuted_means.append(
            np.mean(
                delta_by_repeat
                * signs
            )
        )


    permuted_means = np.asarray(
        permuted_means
    )


    permutation_p_value = np.mean(
        np.abs(permuted_means)
        >= observed_difference
    )


    # ------------------------------------------------------------------------
    # Save region results
    # ------------------------------------------------------------------------

    ablation_statistics.append({
        "Region removed": region,
        "Lower wavelength": lower,
        "Upper wavelength": upper,
        "Bands removed": bands_removed,
        "Full-spectrum F1": mean_full_f1,
        "Ablated F1": mean_reduced_f1,
        "Delta F1": mean_delta_f1,
        "95% CI lower": confidence_interval_lower,
        "95% CI upper": confidence_interval_upper,
        "Permutation p-value": permutation_p_value,
    })


    print(
        f"  Full F1    = {mean_full_f1:.4f}"
    )

    print(
        f"  Ablated F1 = {mean_reduced_f1:.4f}"
    )

    print(
        f"  ΔF1        = {mean_delta_f1:+.4f}"
    )

    print(
        "  95% CI     = "
        f"[{confidence_interval_lower:+.4f}, "
        f"{confidence_interval_upper:+.4f}]"
    )

    print(
        f"  p          = {permutation_p_value:.4f}"
    )


# ----------------------------------------------------------------------------
# Convert results to a table
# ----------------------------------------------------------------------------

ablation_statistics = pd.DataFrame(
    ablation_statistics
)


# ----------------------------------------------------------------------------
# Correct for testing five spectral regions
#
# Holm correction controls the family-wise error rate while being less
# conservative than a simple Bonferroni correction.
# ----------------------------------------------------------------------------

ablation_statistics[
    "Holm-adjusted p-value"
] = multipletests(
    ablation_statistics[
        "Permutation p-value"
    ],
    method="holm",
)[1]


# ----------------------------------------------------------------------------
# Indicate whether the bootstrap interval includes zero
# ----------------------------------------------------------------------------

ablation_statistics[
    "95% CI excludes zero"
] = (
    (
        ablation_statistics[
            "95% CI lower"
        ] > 0
    )
    |
    (
        ablation_statistics[
            "95% CI upper"
        ] < 0
    )
)


# ----------------------------------------------------------------------------
# Sort the table in spectral order
# ----------------------------------------------------------------------------

region_order = [
    "Visible",
    "Red-edge",
    "NIR",
    "SWIR-1",
    "SWIR-2",
]


ablation_statistics[
    "Region removed"
] = pd.Categorical(
    ablation_statistics[
        "Region removed"
    ],
    categories=region_order,
    ordered=True,
)


ablation_statistics = (
    ablation_statistics
    .sort_values(
        "Region removed"
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------------------------
# Display publication-ready results
# ----------------------------------------------------------------------------

print(
    "\n\n"
    "============================================================"
)

print(
    "LEAVE-ONE-REGION-OUT ABLATION RESULTS"
)

print(
    "============================================================\n"
)


display_columns = [
    "Region removed",
    "Full-spectrum F1",
    "Ablated F1",
    "Delta F1",
    "95% CI lower",
    "95% CI upper",
    "Permutation p-value",
    "Holm-adjusted p-value",
    "95% CI excludes zero",
]


print(
    ablation_statistics[
        display_columns
    ]
    .round(4)
    .to_string(index=False)
)


# ----------------------------------------------------------------------------
# Save complete results
# ----------------------------------------------------------------------------

ablation_statistics.to_csv(
    TABLE_DIR
    / "leave_one_region_out_ablation_statistics.csv",
    index=False,
)


# ----------------------------------------------------------------------------
# Also save the repeat-level paired scores
#
# These are useful for Supporting Information and reproducibility.
# ----------------------------------------------------------------------------

repeat_level_results = []


for region, reduced_scores in ablation_fold_scores.items():

    reduced_scores_by_repeat = reduced_scores.reshape(
        ABLATION_REPEATS,
        CV_FOLDS,
    ).mean(axis=1)


    for repeat_number in range(
        ABLATION_REPEATS
    ):

        repeat_level_results.append({
            "Region removed": region,
            "Repeat": repeat_number + 1,
            "Full-spectrum F1":
                full_scores_by_repeat[
                    repeat_number
                ],
            "Ablated F1":
                reduced_scores_by_repeat[
                    repeat_number
                ],
            "Delta F1":
                reduced_scores_by_repeat[
                    repeat_number
                ]
                - full_scores_by_repeat[
                    repeat_number
                ],
        })


repeat_level_results = pd.DataFrame(
    repeat_level_results
)


repeat_level_results.to_csv(
    TABLE_DIR
    / "leave_one_region_out_repeat_level_scores.csv",
    index=False,
)


print(
    "\nSaved:"
)

print(
    TABLE_DIR
    / "leave_one_region_out_ablation_statistics.csv"
)

print(
    TABLE_DIR
    / "leave_one_region_out_repeat_level_scores.csv"
)

10 repeats × 5 folds = 50 paired folds
Classifier used: XGBoost

Running full-spectrum baseline...
Full-spectrum mean F1 = 0.5810

Removing Visible: 450–700 nm
  Bands removed   : 50
  Bands remaining : 274
  Full F1    = 0.5810
  Ablated F1 = 0.5499
  ΔF1        = -0.0311
  95% CI     = [-0.0507, -0.0100]
  p          = 0.0254

Removing Red-edge: 700–780 nm
  Bands removed   : 16
  Bands remaining : 308
  Full F1    = 0.5810
  Ablated F1 = 0.5602
  ΔF1        = -0.0207
  95% CI     = [-0.0384, +0.0002]
  p          = 0.0781

Removing NIR: 780–1340 nm
  Bands removed   : 112
  Bands remaining : 212
  Full F1    = 0.5810
  Ablated F1 = 0.5629
  ΔF1        = -0.0181
  95% CI     = [-0.0457, +0.0122]
  p          = 0.2754

Removing SWIR-1: 1480–1800 nm
  Bands removed   : 63
  Bands remaining : 261
  Full F1    = 0.5810
  Ablated F1 = 0.5761
  ΔF1        = -0.0048
  95% CI     = [-0.0348, +0.0247]
  p          = 0.7559

Removing SWIR-2: 1980–2400 nm
  Bands removed   : 83
  Bands remainin

In [34]:
# Run the SHAP analysis for XGBoost if the package is available.

shap_wavelength = None
shap_pc = None

if HAVE_SHAP:
    print("Running XGBoost wavelength attribution ...")

    # Fit PCA on the complete dataset only for attribution.
    # This is descriptive model interpretation, not an independent performance estimate.
    pca_attribution = PCA(
        n_components=n_pcs,
        random_state=RANDOM_STATE,
    )

    Z_attribution = pca_attribution.fit_transform(X_class)

    xgb_attribution = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    xgb_attribution.fit(Z_attribution, y_encoded)

    # TreeExplainer can return either an array or a list depending on SHAP version.
    explainer = shap.TreeExplainer(xgb_attribution)
    shap_values = explainer.shap_values(Z_attribution)

    if isinstance(shap_values, list):
        shap_target = shap_values[target_encoded]
    else:
        shap_values = np.asarray(shap_values)

        if shap_values.ndim == 3:
            # Common multiclass shape: observations × PCs × classes.
            shap_target = shap_values[:, :, target_encoded]
        else:
            shap_target = shap_values

    shap_pc = np.abs(shap_target).mean(axis=0)
    shap_pc = shap_pc / shap_pc.max() if shap_pc.max() > 0 else shap_pc

    # Reconstruct a wavelength-oriented importance score from the PCA loadings.
    loading_weights = np.abs(pca_attribution.components_)
    reconstructed = (shap_pc[:, None] * loading_weights).sum(axis=0)
    reconstructed = (
        reconstructed / reconstructed.max()
        if reconstructed.max() > 0
        else reconstructed
    )

    shap_wavelength = pd.DataFrame({
        "wavelength_nm": wl_good,
        "SHAP_reconstructed": reconstructed,
    })

    shap_wavelength.to_csv(
        TABLE_DIR / "SHAP_wavelength_attribution.csv",
        index=False,
    )

    pd.DataFrame({
        "PC": [f"PC{i + 1}" for i in range(n_pcs)],
        "mean_abs_SHAP": shap_pc,
        "variance_explained": pca_attribution.explained_variance_ratio_,
    }).to_csv(
        TABLE_DIR / "SHAP_PC_attribution.csv",
        index=False,
    )

    print("Top reconstructed wavelengths:")
    print(
        shap_wavelength.nlargest(10, "SHAP_reconstructed")
        .round(4)
        .to_string(index=False)
    )

else:
    print(
        "SHAP is not installed. The notebook can still complete the spectral "
        "profile, SAM, PCA, classification, bootstrap and ablation analyses."
    )

Running XGBoost wavelength attribution ...
Top reconstructed wavelengths:
 wavelength_nm  SHAP_reconstructed
        1795.0              1.0000
        1123.0              0.8969
        1128.0              0.8659
         763.0              0.8324
         768.0              0.8289
         758.0              0.8282
         773.0              0.8257
         778.0              0.8157
         753.0              0.8143
         783.0              0.8020


In [39]:
# Figure 6 — Spectral basis of L. argenteum discrimination.
#
# Panel (a): Mean spectral profiles of L. argenteum and its closest
#            spectral confusers, with 95% confidence intervals.
#
# Panel (b): Leave-one-region-out ablation analysis showing mean ΔF1,
#            bootstrap 95% confidence intervals and Holm-adjusted p-values.
#
# Panel (c): Reconstructed wavelength-level SHAP importance from XGBoost.


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.transforms import blended_transform_factory


# ============================================================================
# Figure appearance
# ============================================================================

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 400,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.12,
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


fig, axes = plt.subplots(
    3,
    1,
    figsize=(8.6, 13.1),
    gridspec_kw={
        "height_ratios": [
            1.35,
            1.25,
            1.05,
        ],
    },
)


# ============================================================================
# Panel (a): target species against spectral confusers
# ============================================================================

ax = axes[0]


for i, species in enumerate(confusers):

    values = vn_df.loc[
        vn_df["Species"] == species,
        band_cols,
    ].to_numpy(float)


    mean_values = values.mean(
        axis=0
    )


    se = (
        values.std(
            axis=0,
            ddof=1,
        )
        / np.sqrt(
            values.shape[0]
        )
    )


    colour = plt.get_cmap(
        "viridis"
    )(
        0.30
        + 0.50
        * i
        / max(
            len(confusers) - 1,
            1,
        )
    )


    ax.fill_between(
        wl_good,
        mean_values - 1.96 * se,
        mean_values + 1.96 * se,
        color=colour,
        alpha=0.16,
        linewidth=0,
    )


    ax.plot(
        wl_good,
        mean_values,
        color=colour,
        linewidth=1.0,
        label=species,
    )


# Plot L. argenteum last so that the target spectrum remains visible.

target_values = vn_df.loc[
    vn_df["Species"] == TARGET,
    band_cols,
].to_numpy(float)


target_mean = target_values.mean(
    axis=0
)


target_se = (
    target_values.std(
        axis=0,
        ddof=1,
    )
    / np.sqrt(
        target_values.shape[0]
    )
)


ax.fill_between(
    wl_good,
    target_mean - 1.96 * target_se,
    target_mean + 1.96 * target_se,
    color=TARGET_BLUE,
    alpha=0.16,
    linewidth=0,
)


ax.plot(
    wl_good,
    target_mean,
    color=TARGET_BLUE,
    linewidth=1.8,
    label=TARGET,
)


# Shade the atmospheric water-absorption regions excluded from analysis.

for lower, upper in [
    (1340, 1480),
    (1800, 1980),
]:

    ax.axvspan(
        lower,
        upper,
        color=WATER_GREY,
        alpha=0.50,
        zorder=0,
    )


ax.set_xlim(
    float(
        np.min(
            wl_good
        )
    ),
    float(
        np.max(
            wl_good
        )
    ),
)


ax.set_xlabel(
    "Wavelength (nm)"
)

ax.set_ylabel(
    "Vector-normalised reflectance"
)


ax.set_title(
    "(a) Target versus spectral confusers",
    loc="left",
    fontweight="bold",
    pad=7,
)


legend = ax.legend(
    frameon=False,
    fontsize=6.8,
    ncol=2,
)


for text in legend.get_texts():

    text.set_fontstyle(
        "italic"
    )


# ============================================================================
# Panel (b): leave-one-region-out ablation
# ============================================================================

ax = axes[1]


# Use the actual results table created in the preceding ablation analysis.
ablation_plot = (
    ablation_statistics
    .copy()
)


# Make sure the plotting columns are numeric.
numeric_columns = [
    "Full-spectrum F1",
    "Ablated F1",
    "Delta F1",
    "95% CI lower",
    "95% CI upper",
    "Permutation p-value",
    "Holm-adjusted p-value",
]


for column in numeric_columns:

    ablation_plot[
        column
    ] = pd.to_numeric(
        ablation_plot[
            column
        ],
        errors="coerce",
    )


ablation_plot = ablation_plot.dropna(
    subset=[
        "Delta F1",
        "95% CI lower",
        "95% CI upper",
        "Holm-adjusted p-value",
    ]
)


if ablation_plot.empty:

    ax.text(
        0.5,
        0.5,
        "No regional ablation results available",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=9,
    )

else:

    # -----------------------------------------------------------------------
    # Sort by ΔF1 for visual interpretation.
    #
    # Negative ΔF1 means performance decreased after removing that region,
    # indicating that the region contributed useful information.
    # -----------------------------------------------------------------------

    ablation_plot = (
        ablation_plot
        .sort_values(
            "Delta F1",
            ascending=True,
        )
        .reset_index(drop=True)
    )


    region_names = (
        ablation_plot[
            "Region removed"
        ]
        .astype(str)
        .to_numpy()
    )


    delta_values = (
        ablation_plot[
            "Delta F1"
        ]
        .to_numpy(float)
    )


    ci_lower = (
        ablation_plot[
            "95% CI lower"
        ]
        .to_numpy(float)
    )


    ci_upper = (
        ablation_plot[
            "95% CI upper"
        ]
        .to_numpy(float)
    )


    adjusted_p_values = (
        ablation_plot[
            "Holm-adjusted p-value"
        ]
        .to_numpy(float)
    )


    ci_excludes_zero = (
        ablation_plot[
            "95% CI excludes zero"
        ]
        .astype(bool)
        .to_numpy()
    )


    y_positions = np.arange(
        len(
            region_names
        )
    )


    # Convert absolute CI limits into distances from the ΔF1 estimate.
    lower_error = (
        delta_values
        - ci_lower
    )

    upper_error = (
        ci_upper
        - delta_values
    )

    lower_error = np.maximum(
        lower_error,
        0,
    )

    upper_error = np.maximum(
        upper_error,
        0,
    )


    # Orange = removing the region reduced F1.
    # Green  = removing the region increased F1.
    point_colours = np.where(
        delta_values < 0,
        "#D55E00",
        "#009E73",
    )


    # Plot ΔF1 point estimates and 95% CIs.
    for (
        y_position,
        delta_value,
        lower,
        upper,
        colour,
        significant_ci,
    ) in zip(
        y_positions,
        delta_values,
        lower_error,
        upper_error,
        point_colours,
        ci_excludes_zero,
    ):

        ax.errorbar(
            delta_value,
            y_position,
            xerr=np.array([
                [lower],
                [upper],
            ]),
            fmt="o",
            markersize=7 if significant_ci else 6,
            color=colour,
            ecolor=colour,
            elinewidth=1.4,
            capsize=3.5,
            capthick=1.0,
            markeredgecolor=(
                "black"
                if significant_ci
                else "white"
            ),
            markeredgewidth=(
                1.0
                if significant_ci
                else 0.7
            ),
            zorder=3,
        )


    # Zero = no change relative to the full-spectrum model.
    ax.axvline(
        0,
        color="#555555",
        linewidth=0.9,
        linestyle="--",
        zorder=1,
    )


    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        region_names
    )


    # Set horizontal limits using the confidence intervals,
    # while reserving extra room for the p-value column.
    minimum_x = float(
        np.min(
            ci_lower
        )
    )

    maximum_x = float(
        np.max(
            ci_upper
        )
    )

    x_range = max(
        maximum_x - minimum_x,
        0.03,
    )

    left_padding = (
        x_range * 0.18
    )

    right_padding = (
        x_range * 0.52
    )

    ax.set_xlim(
        minimum_x - left_padding,
        maximum_x + right_padding,
    )


    # Add ΔF1 values just above the point estimates.
    for (
        y_position,
        delta_value,
    ) in zip(
        y_positions,
        delta_values,
    ):

        ax.annotate(
            f"{delta_value:+.3f}",
            xy=(
                delta_value,
                y_position,
            ),
            xytext=(
                0,
                8,
            ),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7.2,
            color="#333333",
        )


    # Put p-values in a fixed right-hand column using a blended transform:
    # x is in axes coordinates, y is in data coordinates.
    p_value_transform = blended_transform_factory(
        ax.transAxes,
        ax.transData,
    )


    for (
        y_position,
        adjusted_p,
    ) in zip(
        y_positions,
        adjusted_p_values,
    ):

        if adjusted_p < 0.001:

            p_text = "pHolm < 0.001"

        else:

            p_text = f"pHolm = {adjusted_p:.3f}"


        ax.text(
            0.965,
            y_position,
            p_text,
            transform=p_value_transform,
            ha="right",
            va="center",
            fontsize=7.4,
            color="#333333",
        )


    # Put the full-spectrum baseline in the upper-left corner.
    baseline_f1 = float(
        ablation_plot[
            "Full-spectrum F1"
        ].iloc[0]
    )

    ax.text(
        0.02,
        0.96,
        f"Full-spectrum F1 = {baseline_f1:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        color="#333333",
    )


ax.set_xlabel(
    "Change in target F1 (ΔF1)"
)

ax.set_title(
    "(b) Leave-one-region-out ablation",
    loc="left",
    fontweight="bold",
    pad=7,
)


# ============================================================================
# Panel (c): XGBoost wavelength attribution
# ============================================================================

ax = axes[2]


if shap_wavelength is None:

    ax.text(
        0.5,
        0.5,
        "SHAP unavailable",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

else:

    wavelength = (
        shap_wavelength[
            "wavelength_nm"
        ]
        .to_numpy(float)
    )

    importance = (
        shap_wavelength[
            "SHAP_reconstructed"
        ]
        .to_numpy(float)
    )


    ax.plot(
        wavelength,
        importance,
        color="#0072B2",
        linewidth=1.25,
        zorder=2,
    )

    ax.fill_between(
        wavelength,
        0,
        importance,
        color="#56B4E9",
        alpha=0.20,
        linewidth=0,
    )


    # Find local peaks separated by at least 50 nm.
    peak_indices = []

    minimum_peak_distance_nm = 50.0


    for i in range(
        1,
        len(
            importance
        ) - 1,
    ):

        is_peak = (
            importance[i]
            > importance[i - 1]
            and importance[i]
            >= importance[i + 1]
        )

        if not is_peak:
            continue


        if not peak_indices:

            peak_indices.append(
                i
            )

        else:

            previous_i = (
                peak_indices[-1]
            )

            wavelength_gap = (
                wavelength[i]
                - wavelength[
                    previous_i
                ]
            )

            if (
                wavelength_gap
                >= minimum_peak_distance_nm
            ):

                peak_indices.append(
                    i
                )

            elif (
                importance[i]
                > importance[
                    previous_i
                ]
            ):

                peak_indices[-1] = i


    ranked_indices = np.argsort(
        importance
    )[::-1]

    selected_peak_indices = list(
        peak_indices
    )


    for candidate in ranked_indices:

        if (
            len(
                selected_peak_indices
            )
            >= 5
        ):
            break


        candidate_wavelength = (
            wavelength[
                candidate
            ]
        )

        sufficiently_separated = all(
            abs(
                candidate_wavelength
                - wavelength[index]
            ) >= minimum_peak_distance_nm
            for index in selected_peak_indices
        )

        if sufficiently_separated:

            selected_peak_indices.append(
                int(
                    candidate
                )
            )


    selected_peak_indices = sorted(
        selected_peak_indices,
        key=lambda index: importance[
            index
        ],
        reverse=True,
    )[:5]

    selected_peak_indices = sorted(
        selected_peak_indices,
        key=lambda index: wavelength[
            index
        ],
    )


    for (
        label_number,
        index,
    ) in enumerate(
        selected_peak_indices
    ):

        wavelength_value = float(
            wavelength[
                index
            ]
        )

        importance_value = float(
            importance[
                index
            ]
        )


        ax.scatter(
            wavelength_value,
            importance_value,
            s=28,
            color="#0072B2",
            edgecolor="white",
            linewidth=0.8,
            zorder=4,
        )


        vertical_offset = (
            10
            if label_number % 2 == 0
            else 24
        )


        ax.annotate(
            f"{wavelength_value:.0f} nm",
            xy=(
                wavelength_value,
                importance_value,
            ),
            xytext=(
                0,
                vertical_offset,
            ),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7.5,
            arrowprops={
                "arrowstyle": "-",
                "color": "#555555",
                "linewidth": 0.7,
            },
        )


    for lower, upper in [
        (1340, 1480),
        (1800, 1980),
    ]:

        ax.axvspan(
            lower,
            upper,
            color=WATER_GREY,
            alpha=0.50,
            zorder=0,
        )


    ax.set_ylabel(
        "Reconstructed SHAP importance"
    )


ax.set_xlabel(
    "Wavelength (nm)"
)

ax.set_title(
    "(c) XGBoost wavelength attribution",
    loc="left",
    fontweight="bold",
    pad=7,
)


# ============================================================================
# Wavelength limits for spectral panels only
# ============================================================================

axes[0].set_xlim(
    float(
        np.min(
            wl_good
        )
    ),
    float(
        np.max(
            wl_good
        )
    ),
)

axes[2].set_xlim(
    float(
        np.min(
            wl_good
        )
    ),
    float(
        np.max(
            wl_good
        )
    ),
)


# ============================================================================
# Final layout
# ============================================================================

fig.subplots_adjust(
    left=0.11,
    right=0.97,
    bottom=0.065,
    top=0.97,
    hspace=0.42,
)


# ============================================================================
# Save manuscript figure
# ============================================================================

fig.savefig(
    FIG_DIR
    / "Fig6_spectral_basis.png"
)

fig.savefig(
    FIG_DIR
    / "Fig6_spectral_basis.pdf"
)


plt.show()


# ============================================================================
# Print wavelengths labelled in panel (c)
# ============================================================================

if shap_wavelength is not None:

    print(
        "\nTop reconstructed wavelengths shown in Figure 6c:"
    )

    selected_output = []

    for index in selected_peak_indices:

        selected_output.append({
            "Wavelength (nm)": wavelength[index],
            "Reconstructed importance": importance[index],
        })

    print(
        pd.DataFrame(
            selected_output
        )
        .round(4)
        .to_string(index=False)
    )


Top reconstructed wavelengths shown in Figure 6c:
 Wavelength (nm)  Reconstructed importance
           698.0                    0.7648
           763.0                    0.8324
          1123.0                    0.8969
          1654.0                    0.6994
          1795.0                    1.0000


## 11. Final performance table and analysis audit

The final table below collects the numbers that should be carried into the manuscript. The point of keeping this table at the end of the notebook is that it is generated from the same objects used to make the figures; there is no separate data-loading route for the manuscript numbers.

In [ ]:
# Final manuscript-ready summary.

target_row = metrics_df.loc[metrics_df["Model"] == BEST_MODEL].iloc[0]

headline = pd.DataFrame({
    "Quantity": [
        "Retained spectral bands",
        "Spectra used",
        "Species used",
        "Best classifier",
        "Target sensitivity",
        "Target precision",
        "Target F1",
        "Overall accuracy",
        "Target F1 bootstrap 95% CI lower",
        "Target F1 bootstrap 95% CI upper",
        "Target rank by F1",
        "Nearest spectral neighbour",
        "Nearest spectral angle (degrees)",
    ],
    "Value": [
        len(wl_good),
        len(df_cv),
        len(eligible_species),
        BEST_MODEL,
        target_row["Target sensitivity"],
        target_row["Target precision"],
        target_row["Target F1"],
        target_row["Overall accuracy"],
        f1_ci[0],
        f1_ci[1],
        target_rank,
        target_angles.index[0],
        target_angles.iloc[0],
    ],
})

headline.to_csv(
    TABLE_DIR / "headline_results_324bands.csv",
    index=False,
)

print(headline.to_string(index=False))

print("\nFigures written:")
for figure_file in sorted(FIG_DIR.glob("*")):
    print(f"  {figure_file.name}")

print("\nTables written:")
for table_file in sorted(TABLE_DIR.glob("*.csv")):
    print(f"  {table_file.name}")

## 12. Interpretation notes for the manuscript

The statistical outputs from this notebook should be interpreted in the same conceptual order as the analyses themselves. The spectral profile figure describes the observed species-level spectral shapes. SAM quantifies pairwise similarity but does not by itself establish successful classification. PCA reduces the predictor space. The classifier comparison then evaluates whether those spectral predictors permit the species to be distinguished under cross-validation.

The bootstrap interval belongs to the out-of-fold performance estimate of the selected classifier. It should not be described as evidence from an independent field validation dataset. Likewise, a low overall multi-species accuracy can coexist with a comparatively high target-species F1-score because the latter answers the narrower monitoring question: how well is *L. argenteum* separated from the other sampled taxa.

No LiDAR height, canopy geometry, spatial prediction, raster masking or probability mapping is used anywhere in this notebook. Those analyses remain downstream of the spectral classification work.